# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed0607/ML-Internship-Strter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## **Paper Finding 1: Refresh Impact on Organic Visibility**

  Finding: Updated content exhibits an average lift in impressions compared to unrefreshed historical baselines.

  Methodology Question: What is the exact counterfactual or control design? Without an explicit A/B test or synthetic control group, how much of this observed lift is driven by broader domain-level momentum, seasonality, or sitewide crawl budget changes rather than the content edit itself?

## **Paper Finding 2: AI Referral Traffic Concentration**

  Finding: AI-driven referral sessions are heavily concentrated among top-tier, long-form content assets.

  Methodology Question: How was sparsity and panel imbalance handled? Given that AI referral sessions represent fewer than 0.1% of daily fact rows, were these patterns validated across diverse client cohorts, or is the concentration metric disproportionately influenced by a few high-volume domains with early GA4 tracking?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics.pairwise import euclidean_distances

# 1. Load Dataset
df = pd.read_csv('content_refresh_anonymized.csv')

# Preprocessing & Gotcha Handling
# Ensure numeric columns are actually numeric (coercing any weird strings to NaN, then 0)
df['avg_position'] = pd.to_numeric(df['avg_position'], errors='coerce').fillna(0)
df['word_count'] = pd.to_numeric(df['word_count'], errors='coerce')
df['ctr'] = pd.to_numeric(df['ctr'], errors='coerce').fillna(0)
df['impressions_90d'] = pd.to_numeric(df['impressions_90d'], errors='coerce').fillna(0)
df['sessions_90d'] = pd.to_numeric(df['sessions_90d'], errors='coerce').fillna(0)
df['content_age_days'] = pd.to_numeric(df['content_age_days'], errors='coerce').fillna(0)

# Create clean features
df['has_avg_position'] = (df['avg_position'] > 0).astype(int)
median_pos = df[df['avg_position'] > 0]['avg_position'].median()
# If median_pos is somehow NaN, fallback to 10
if pd.isna(median_pos): median_pos = 10.0
df['avg_position_clean'] = df['avg_position'].replace(0, median_pos)

df['has_word_count'] = df['word_count'].notnull().astype(int)
df['word_count_clean'] = df['word_count'].fillna(0)

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# NEW FIX: Ensure client_id has no empty strings causing the group split to crash
df['client_id'] = df['client_id'].fillna('unknown_client')

features = ['impressions_90d', 'sessions_90d', 'content_age_days',
            'avg_position_clean', 'has_avg_position', 'word_count_clean',
            'has_word_count', 'ctr']

# NEW FIX: Explicitly cast X to float to guarantee no object arrays
X = df[features].copy().astype(float)
y = df['is_declining_label']

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- SPLIT 1: Naive Random Split (Vulnerable to Client Leakage) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

kmeans_r = KMeans(n_clusters=4, random_state=42, n_init=10)
train_clusters_r = kmeans_r.fit_predict(X_train_r_scaled)
risk_cluster_r = pd.Series(y_train_r).groupby(train_clusters_r).mean().idxmax()

dist_r = euclidean_distances(X_test_r_scaled, kmeans_r.cluster_centers_)
risk_score_r = -dist_r[:, risk_cluster_r]
p50_random = precision_at_k(risk_score_r, y_test_r, k=50)

# --- SPLIT 2: Honest Grouped Split (Strict Client Holdout) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=df['client_id']))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

scaler_g = StandardScaler()
X_train_g_scaled = scaler_g.fit_transform(X_train_g)
X_test_g_scaled = scaler_g.transform(X_test_g)

kmeans_g = KMeans(n_clusters=4, random_state=42, n_init=10)
train_clusters_g = kmeans_g.fit_predict(X_train_g_scaled)
risk_cluster_g = pd.Series(y_train_g.values).groupby(train_clusters_g).mean().idxmax()

dist_g = euclidean_distances(X_test_g_scaled, kmeans_g.cluster_centers_)
risk_score_g = -dist_g[:, risk_cluster_g]
p50_grouped = precision_at_k(risk_score_g, y_test_g.values, k=50)

print("--- VALIDATION DESIGN COMPARISON ---")
print(f"Base Rate (Random Split Test):   {y_test_r.mean():.3f}")
print(f"Precision@50 (Random Split):     {p50_random:.3f}")
print(f"Base Rate (Grouped Split Test):  {y_test_g.mean():.3f}")
print(f"Precision@50 (Grouped Split):    {p50_grouped:.3f}")

--- VALIDATION DESIGN COMPARISON ---
Base Rate (Random Split Test):   0.555
Precision@50 (Random Split):     0.700
Base Rate (Grouped Split Test):  0.440
Precision@50 (Grouped Split):    0.220


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## **Leakage Checklist & Audit**

1.   Temporal / Future Leakage: Verified clean. Features are strictly computed over trailing historical windows (impressions_90d, sessions_90d) and CMS creation timestamps (content_age_days).
2.   Target / Derived Leakage: Verified clean. trend_direction and trend_pct were explicitly excluded from the feature matrix $X$.
3.   Product Flag Leakage: Verified clean. No rule outputs (health_score, priority_score, refresh_tier) were included in training.
4.   Client Overlap Leakage: Resolved by adopting GroupShuffleSplit on client_id, ensuring the test set contains only unseen domains.

## **Failure Cases**
*   Failure 1 (False Positive on High-Volume Evergreen): Pages with massive impression counts and mature age get pulled toward the "At-Risk" cluster centroid due to standard variance scaling, even when their rankings are steady.
*   Failure 2 (False Negative on Thin Decaying Pages): Low-volume pages experiencing sharp ranking drops often land in a broad "sparse" cluster rather than the active decline cluster.



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Claim Rewrite Audit

Claim 1: Causal SEO Impact

*    Overstated Claim (Avoid): "The K-Means model proves that older pages cause ranking loss in Google's search algorithm."

*    Honest, Defensible Rewrite: "In the observed snapshot, older pages correlate directionally with higher variance in search visibility, but this does not constitute causal proof of search algorithm mechanics."

Claim 2: Predictive Guarantees

*    Overstated Claim (Avoid): "Our model predicts with high accuracy which pages will recover after an update."

*    Honest, Defensible Rewrite: "The clustering model provides an empirical decision-support ranking that prioritizes pages exhibiting structural and behavioral patterns similar to historically declining content."

Claim 3: Algorithm Capabilities

*    Overstated Claim (Avoid): "The algorithm performs semantic grouping across topics."

*    Honest, Defensible Rewrite: "The model performs metric-based structural grouping using observable performance and metadata signals, not semantic text embeddings."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.